# Aegis Health — SFT training v2 (gentler hyperparameters)

**Why a v2?** Held-out eval on `sft_train.ipynb` v1 showed the model hallucinates and over-defers — classic LoRA overfit from aggressive hyperparameters on a 1518-example dataset. v2 addresses each contributing factor:

| Hyperparam | v1 (broken) | v2 (this) |
|---|---|---|
| `r` | 16 | **8** |
| `lora_alpha` | 32 | **16** (keeps 2:1 ratio with `r`) |
| `num_train_epochs` | 3 | **1** |
| `learning_rate` | 2e-4 | **5e-5** |
| `target_modules` | q/k/v/o_proj only | **+ gate/up/down_proj** (now MLP too) |

Intuition: lower `r` + fewer epochs + lower LR = less capacity to memorize the training format. Adding MLP modules gives the model somewhere to store the DrugSafe format knowledge that *isn't* attention, so it doesn't hijack attention patterns to express it.

**Runtime:** ~20–30 min on A100 (single epoch × ~95 steps).

**Prerequisites:**
- Colab Secret `HF_TOKEN` set
- Empty private HF repos **with v2 suffix** so v1 stays for comparison:
  - `V1rtucious/aegis-sft-e4b-lora-v2`
  - `V1rtucious/aegis-sft-e4b-merged-v2`
- `combined_sft.jsonl` ready to upload (same file as v1)
- Runtime: **Runtime → Change runtime type → GPU (A100 preferred)**

## Cell 1 — SFT deps (same as v1 — correct Unsloth pins)

In [ ]:
!pip install -q -U unsloth unsloth_zoo
!pip install -q -U \
    "trl>=0.18.2,!=0.19.0,<=0.24.0" \
    "peft>=0.12" \
    "datasets>=3.4.1,!=4.0.0,!=4.1.0,<4.4.0" \
    bitsandbytes accelerate

import unsloth, trl, peft, datasets, bitsandbytes, torch, transformers

def _ver(v):
    return tuple(int(x) for x in v.split(".")[:2] if x.isdigit())

assert _ver(trl.__version__) >= (0, 18), f"TRL {trl.__version__} too old"
assert _ver(datasets.__version__) >= (3, 4), f"datasets {datasets.__version__} too old"
assert _ver(torch.__version__) < (2, 11), f"torch {torch.__version__} too new"
assert torch.cuda.is_available(), "No CUDA — Runtime → GPU."

print(f"unsloth {unsloth.__version__} · trl {trl.__version__} · transformers {transformers.__version__}")
print(f"torch {torch.__version__} on {torch.cuda.get_device_name(0)}  bf16={torch.cuda.is_bf16_supported()}")

## Cell 2 — Secrets + HF login

In [ ]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

## Cell 3 — Load Gemma 4 E4B with v2 LoRA config

Key differences from v1:
- `r=8` (was 16) — half the capacity, less prone to memorization
- `lora_alpha=16` (was 32) — keeps `alpha/r = 2` scaling
- `target_modules` now includes MLP layers (`gate_proj`, `up_proj`, `down_proj`) alongside attention. This gives the model **somewhere other than attention to store task-specific knowledge**.

In [ ]:
import os
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="google/gemma-4-e4b-it",
    max_seq_length=2048,
    load_in_4bit=True,
    token=os.environ["HF_TOKEN"],
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    lora_dropout=0,                                        # keeps Unsloth's fast path
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",            # attention
        "gate_proj", "up_proj", "down_proj",               # MLP (new in v2)
    ],
    use_gradient_checkpointing="unsloth",
)

## Cell 4 — Load + format training data (same as v1)

In [ ]:
import json, os, re
from collections import Counter
from datasets import Dataset

# --- Upload combined_sft.jsonl ---
if not os.path.exists("combined_sft.jsonl"):
    from google.colab import files
    print("Select combined_sft.jsonl from your local machine...")
    files.upload()

# --- Upload tool_defs.json ---
TOOL_DEFS_PATH = "tool_defs.json"
if not os.path.exists(TOOL_DEFS_PATH):
    try:
        from huggingface_hub import hf_hub_download
        TOOL_DEFS_PATH = hf_hub_download(
            "V1rtucious/aegis-sft-data", "tool_defs.json",
            repo_type="dataset", local_dir="."
        )
    except Exception:
        from google.colab import files
        print("Upload tool_defs.json (from tools/tools/tool_defs.json)...")
        files.upload()

with open(TOOL_DEFS_PATH) as f:
    TOOL_DEFS_LIST = json.load(f)
print(f"Loaded {len(TOOL_DEFS_LIST)} tool definitions")

records = [json.loads(l) for l in open("combined_sft.jsonl") if l.strip()]
print(f"Loaded {len(records)} records")

_TC_RE = re.compile(r'<\|tool_call>\s*(\{.*?\})\s*<tool_call\|>', re.DOTALL)
_TR_RE = re.compile(r'<\|tool_result>\s*(\{.*?\})\s*<tool_result\|>', re.DOTALL)
_HEALTH_SYMPTOM_RE = re.compile(
    r"\b("
    r"do i have|could this be|is this|am i|diagnos(?:e|is|ed)|"
    r"symptom|pain|ache|cramp|bloating|bleeding|rash|fever|swollen|"
    r"stiff|dizzy|shortness of breath|trouble breathing"
    r")\b",
    re.I,
)
_REQUIRED = {"flags", "citations", "confidence", "defer_to_professional", "explanation"}


def _mode(ex):
    m = (ex.get("mode") or "drugsafe").strip().lower()
    return "consent" if m == "consentreader" else m


def _is_health_symptom(ex):
    if _mode(ex) != "healthpartner":
        return False
    if ex.get("category") == "safety_diagnosis":
        return True
    user_text = " ".join(
        str(m.get("content", ""))
        for m in ex.get("conversation", [])
        if isinstance(m, dict) and m.get("role") == "user"
    )
    return bool(_HEALTH_SYMPTOM_RE.search(user_text))


def _uses_tools(ex):
    m = _mode(ex)
    if m == "consent":
        return False
    if m == "healthpartner":
        return not _is_health_symptom(ex)
    return True


def _strip_tool_defs(text):
    return text.split("Available tools:", 1)[0].strip() if "Available tools:" in text else text.strip()


def _strip_tool_blocks(text):
    return _TC_RE.sub("", _TR_RE.sub("", text)).strip()


def _extract_final_json(ex):
    for msg in reversed(ex.get("conversation", [])):
        if msg.get("role") not in ("model", "assistant"):
            continue
        text = _strip_tool_blocks(msg.get("content", ""))
        try:
            parsed = json.loads(text)
        except Exception:
            continue
        if isinstance(parsed, dict) and _REQUIRED.issubset(parsed):
            return json.dumps(parsed, ensure_ascii=False, separators=(",", ":"))
    raise ValueError("No valid final AegisResponse JSON found")


def _tool_results(content):
    out = []
    for match in _TR_RE.finditer(content):
        parsed = json.loads(match.group(1))
        out.append({"name": parsed.get("name", "unknown"), "content": parsed.get("result", parsed)})
    return out


def _tool_calls(content, start_id):
    calls = []
    for match in _TC_RE.finditer(content):
        parsed = json.loads(match.group(1))
        calls.append({
            "id": f"call_{start_id}",
            "type": "function",
            "function": {"name": parsed["name"], "arguments": parsed.get("arguments", {}) or {}},
        })
        start_id += 1
    return calls, start_id


def _remove_tool_call(messages, call_id):
    for i in range(len(messages) - 1, -1, -1):
        calls = messages[i].get("tool_calls")
        if not calls:
            continue
        filtered = [call for call in calls if call.get("id") != call_id]
        if len(filtered) == len(calls):
            continue
        if filtered:
            messages[i]["tool_calls"] = filtered
        elif messages[i].get("content"):
            messages[i].pop("tool_calls", None)
        else:
            messages.pop(i)
        return


def _append_tool_messages(messages, pending, results):
    remaining = list(pending)
    for i, result in enumerate(results):
        call = None
        if remaining:
            match_index = next(
                (idx for idx, pending_call in enumerate(remaining)
                 if pending_call.get("function", {}).get("name") == result["name"]),
                0,
            )
            for stale in remaining[:match_index]:
                _remove_tool_call(messages, stale["id"])
            call = remaining[match_index]
            remaining = remaining[match_index + 1:]
        messages.append({
            "role": "tool",
            "tool_call_id": call["id"] if call else f"unmatched_{i}",
            "name": result["name"],
            "content": result["content"],
        })
    return remaining




def _native_value(value):
    if isinstance(value, str):
        return f'<|"|>{value}<|"|>'
    if isinstance(value, bool):
        return "true" if value else "false"
    if value is None:
        return "null"
    if isinstance(value, (int, float)):
        return str(value)
    if isinstance(value, list):
        return "[" + ",".join(_native_value(v) for v in value) + "]"
    if isinstance(value, dict):
        return "{" + ",".join(f"{k}:{_native_value(v)}" for k, v in sorted(value.items())) + "}"
    return f'<|"|>{str(value)}<|"|>'


def _native_tool_response(message):
    content = message.get("content", {})
    if isinstance(content, dict):
        inner = ",".join(f"{k}:{_native_value(v)}" for k, v in sorted(content.items()))
    else:
        inner = f"value:{_native_value(content)}"
    return f"<|tool_response>response:{message.get('name', 'unknown')}{{{inner}}}<tool_response|>"


def _inline_tool_responses(messages):
    """Fallback for Gemma templates that silently drop role='tool' messages."""
    out = []
    for message in messages:
        if message.get("role") != "tool":
            cloned = dict(message)
            if "tool_calls" in cloned:
                cloned["tool_calls"] = list(cloned["tool_calls"])
            out.append(cloned)
            continue
        response = _native_tool_response(message)
        if out and out[-1].get("role") == "assistant":
            out[-1]["content"] = str(out[-1].get("content", "")) + response
        else:
            out.append({"role": "assistant", "content": response})
    return out

def to_messages(ex):
    raw = ex.get("conversation", [])
    messages = [{"role": "system", "content": _strip_tool_defs(raw[0]["content"])}]

    if not _uses_tools(ex):
        cleaned_user_turns = [
            _strip_tool_blocks(m.get("content", ""))
            for m in raw
            if isinstance(m, dict) and m.get("role") == "user"
        ]
        messages.append({"role": "user", "content": "\n\n".join(t for t in cleaned_user_turns if t)})
        messages.append({"role": "assistant", "content": _extract_final_json(ex)})
        return messages

    pending = []
    next_id = 0
    for msg in raw[1:]:
        role = msg.get("role")
        content = msg.get("content", "")

        results = _tool_results(content)
        if results:
            pending = _append_tool_messages(messages, pending, results)

        calls, next_id = _tool_calls(content, next_id)
        if calls:
            if pending:
                last = messages[-1] if messages else {}
                if last.get("role") == "assistant" and "tool_calls" in last:
                    last["tool_calls"].extend(calls)
                    pending = last["tool_calls"]
                else:
                    messages.append({"role": "assistant", "content": "", "tool_calls": calls})
                    pending.extend(calls)
            else:
                messages.append({"role": "assistant", "content": "", "tool_calls": calls})
                pending = calls

        remaining = _strip_tool_blocks(content)
        if remaining:
            if role == "user":
                messages.append({"role": "user", "content": remaining})
            else:
                parsed = json.loads(remaining)
                if not (isinstance(parsed, dict) and _REQUIRED.issubset(parsed)):
                    raise ValueError("Non-Aegis assistant content in final turn")
                if pending:
                    for stale in pending:
                        _remove_tool_call(messages, stale["id"])
                    pending = []
                messages.append({"role": "assistant", "content": json.dumps(parsed, ensure_ascii=False, separators=(",", ":"))})

    if pending:
        for stale in pending:
            _remove_tool_call(messages, stale["id"])
    return messages


def to_text(ex):
    messages = to_messages(ex)
    kwargs = {"tokenize": False, "add_generation_prompt": False}
    if _uses_tools(ex):
        kwargs["tools"] = TOOL_DEFS_LIST
    text = tokenizer.apply_chat_template(messages, **kwargs)

    has_tool_messages = any(m.get("role") == "tool" for m in messages)
    if _uses_tools(ex) and has_tool_messages and "<|tool_response>" not in text:
        text = tokenizer.apply_chat_template(_inline_tool_responses(messages), **kwargs)

    if '<|tool_call>{' in text or '<|tool_result>' in text or '<tool_result|>' in text:
        raise ValueError("Rendered text leaked legacy tool markers")
    if _uses_tools(ex) and any(_TC_RE.search(m.get("content", "")) for m in ex.get("conversation", [])):
        assert '<|tool_call>call:' in text, "native call: format not found"
    if _uses_tools(ex) and any(_TR_RE.search(m.get("content", "")) for m in ex.get("conversation", [])):
        assert '<|tool_response>' in text, "native tool_response format not found"
    if not _uses_tools(ex):
        assert '<|tool_call>' not in text, "no-tool mode rendered tool calls"
        assert '<|tool_response>' not in text, "no-tool mode rendered tool responses"
    return text


texts = []
policy_counts = Counter()
errors = []
for i, record in enumerate(records, 1):
    try:
        texts.append(to_text(record))
        policy_counts[(_mode(record), "tools" if _uses_tools(record) else "no_tools")] += 1
    except Exception as exc:
        errors.append((i, record.get("mode"), record.get("template"), str(exc)))

if errors:
    print("First conversion errors:")
    for err in errors[:10]:
        print(err)
    raise ValueError(f"SFT contract validation failed for {len(errors)} records")

print(f"Contract policy counts: {dict(policy_counts)}")
ds = Dataset.from_dict({"text": texts}).train_test_split(test_size=0.1, seed=42)
print(f"train={len(ds['train'])}  val={len(ds['test'])}")

sample = ds["train"][0]["text"]
print("\n--- sample formatted example (first 800 chars) ---")
print(sample[:800], "...")
print(f"\n  Contains native 'call:' tool calls: {'<|tool_call>call:' in sample}")
print(f"  Contains legacy JSON tool calls:   {'<|tool_call>{' in sample}")
print(f"  Contains legacy tool_result:       {'<|tool_result>' in sample}")
print(f"  Contains tool catalog:             {'<|tool>' in sample}")
print("Training data rendered with aligned Aegis/Gemma tool contract")


## Cell 5 — Train (gentler: 1 epoch · lr 5e-5)

Effective batch = 4 × 4 = 16. ~1366 train samples / 16 = **~85 optimizer steps** for 1 epoch. Expected wall time: ~20 min on A100. Validation loss should decrease smoothly; if it diverges, stop early.

In [ ]:
from trl import SFTConfig, SFTTrainer
import torch

_use_bf16 = torch.cuda.is_bf16_supported()

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    args=SFTConfig(
        output_dir="/content/sft-e4b-v2",
        dataset_text_field="text",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=1,                      # v1 was 3
        learning_rate=5e-5,                      # v1 was 2e-4
        lr_scheduler_type="cosine",
        warmup_steps=5,
        logging_steps=5,
        save_steps=50,
        eval_strategy="steps",
        eval_steps=20,                           # more frequent eval on shorter run
        optim="adamw_8bit",
        bf16=_use_bf16,
        fp16=not _use_bf16,
        report_to="none",
    ),
)
trainer.train()

## Cell 6 — Save LoRA + push to HF (v2 repo)

**Note the `-v2` suffix.** Keeping v1 on Hub for comparison.

In [ ]:
LORA_REPO = "V1rtucious/aegis-sft-e4b-lora-v2"

model.save_pretrained("/content/lora-v2")
tokenizer.save_pretrained("/content/lora-v2")

model.push_to_hub(LORA_REPO, private=True, token=os.environ["HF_TOKEN"])
tokenizer.push_to_hub(LORA_REPO, private=True, token=os.environ["HF_TOKEN"])
print(f"LoRA v2 pushed: https://huggingface.co/{LORA_REPO}")

## Cell 7 — Merge LoRA → FP16 (~5 min)

In [ ]:
model.save_pretrained_merged(
    "/content/merged-v2",
    tokenizer,
    save_method="merged_16bit",
)
!du -sh /content/merged-v2

## Cell 8 — Push merged FP16 to HF Hub (v2)

~5–10 min for 8 GB. This is what the eval notebook will pull.

In [ ]:
from huggingface_hub import HfApi, create_repo

MERGED_REPO = "V1rtucious/aegis-sft-e4b-merged-v2"

create_repo(MERGED_REPO, private=True, exist_ok=True, token=os.environ["HF_TOKEN"])
HfApi().upload_folder(
    folder_path="/content/merged-v2",
    repo_id=MERGED_REPO,
    repo_type="model",
    token=os.environ["HF_TOKEN"],
)
print(f"Merged v2 pushed: https://huggingface.co/{MERGED_REPO}")
print("\nNext: re-run eval_heldout.ipynb with V1rtucious/aegis-sft-e4b-merged-v2 as the SFT target.")